# 🔧 PRACTICAL: Data-Chain ETL — 7 Async Jobs + BOSS! 👑

**Folder:** 00_Python → 12_Async_Parallel_Python → 03_Async_IO_Basics
**Type:** PRACTICAL · **Data:** self-made mini-CSV (20 rows, self-contained!)

---
## 🎮 Story
The "Sensor-Thief Roadside Diner" picked up a RAW sensor-CSV — 20 readings. The full ETL
chain with *your* async-toolkit: read → process (gather-map!) → throttle on a 2-lane
semaphore → show the slow-sensor the timeout red-card → fire-&-collect with create_task →
the final bill. And finally the BOSS: 10 asserts → **ETL-IN-CHARGE
Badge**! 🏅

**Grading:** ✅ EASY (1-2) · 🟡 MEDIUM (3-5) · 🔴 HARD (6-7) + 👑 BOSS

<details><summary>📦 <b>INTAKE (click)</b></summary>


In [1]:
# 🧺 INTAKE — mini sensor-CSV banao + imports
import asyncio, csv, time
try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

def run(coro):
    return asyncio.run(coro)

with open("etl_rows.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "value"])
    for i in range(20):
        w.writerow([i, i * 5])          # value = 0,5,10,...,95

rows = list(csv.DictReader(open("etl_rows.csv")))
print("rows:", len(rows), "| first:", rows[0], "| last:", rows[-1])
# Expected OUTPUT:
# rows: 20 | first: {'id': '0', 'value': '0'} | last: {'id': '19', 'value': '95'}

rows: 20 | first: {'id': '0', 'value': '0'} | last: {'id': '19', 'value': '95'}


</details>

---
## ✅ Job 1 (EASY): CSV Read + Clean 🧹
`VALS` = [int(r["value"]) for r in rows] — 0..95 step-5!

In [2]:
# ✅ S1 TODO:
# VALS = [int(r["value"]) for r in rows]
# print(VALS[:4], VALS[-3:])

---
## ✅ Job 2 (EASY): A New Processor ⚙️
async process_row(idx, val): sleep(0.01) → dict {idx, scaled=val*0.9,
topic="HIGH" if val>=50 else "LOW"}!

In [3]:
# ✅ S2 TODO:
# async def process_row(idx, val): ...

---
## ✅ Job 3 (MEDIUM): Gather-Map Over ALL 20 🗺️
`PROC` = all rows processed with one single gather (order-preserved!)!

In [4]:
# ✅ S3 TODO:
# PROC = run(asyncio.gather(*(process_row(... , ...) for ...)))

---
## ✅ Job 4 (MEDIUM): Semaphore Lanes 🚦
track `PEAK` with Semaphore(2) — all rows, but 2 active max!

In [5]:
# ✅ S4 TODO:
# PEAK = 0
# async def throttle_all(): nonlocal-style counter badha...

---
## ✅ Job 5 (MEDIUM): Slow-Sensor Red-Card 🛡️
even-index slow (sleep 0.4), odd fast (0.02) — wait_for(0.1) per task:
`TIMEOUTS` list of failed indices (there should be 10!)

In [6]:
# ✅ S5 TODO:
# TIMEOUTS = []
# async def safe_read(i, sleep_t): try wait_for ... except TimeoutError → append i

---
## ✅ Job 6 (HARD): create_task Fire-&-Collect 📝
make the tasks for the first 3 rows → print the done() poll → collect into `TRIO` with await!

In [7]:
# ✅ S6 TODO:
# TRIO = []
# async def collect_first3(): tasks = [create_task(process_row(i, v)) ...]; await each

---
## ✅ Job 7 (HARD): Final Bill 💰
from PROC get `TOT` (sum scaled) and `AVG` — exactly 855.0 & 42.75!

In [8]:
# ✅ S7 TODO:
# TOT = sum(p["scaled"] for p in PROC); AVG = TOT / len(PROC)

---
## 👑 BOSS: 10-Assert ETL-In-Charge Badge (self-contained over INTAKE!) 🏆

In [9]:
# 👑 BOSS CONSOLE
_v = [int(r["value"]) for r in rows]
assert _v == [i * 5 for i in range(20)],                "① csv-clean: 0..95 step-5!"

async def _pr(idx, val):
    await asyncio.sleep(0.01)
    return {"idx": idx, "scaled": val * 0.9, "topic": "HIGH" if val >= 50 else "LOW"}

_p = run(asyncio.gather(*(_pr(i, v) for i, v in enumerate(_v))))
assert len(_p) == 20 and _p[0]["idx"] == 0 and _p[-1]["idx"] == 19, "② gather-map order!"
assert _p[0]["scaled"] == 0.0 and _p[-1]["scaled"] == 85.5,         "③ scaled 0.0 & 85.5!"
assert sum(1 for x in _p if x["topic"] == "HIGH") == 10,            "④ HIGH == 10!"
assert sum(x["scaled"] for x in _p) == 855.0,                       "⑤ total 855.0!"

async def _laned():
    sem = asyncio.Semaphore(2)
    live = 0; peak = 0
    async def one(idx):
        nonlocal live, peak
        async with sem:
            live += 1; peak = max(peak, live)
            await asyncio.sleep(0.01)
            live -= 1
            return idx
    done_ids = await asyncio.gather(*(one(i) for i in range(20)))
    return done_ids, peak
_ids, _peak = run(_laned())
assert _ids == list(range(20)) and _peak == 2,                      "⑥ sem lanes: 20 done, peak 2!"

async def _with_timeouts():
    async def read_one(i):
        async def inner():
            await asyncio.sleep(0.4 if i % 2 == 0 else 0.02)
            return i
        try:
            return await asyncio.wait_for(inner(), timeout=0.1)
        except TimeoutError:
            return None
    return await asyncio.gather(*(read_one(i) for i in range(20)))
_rt = run(_with_timeouts())
assert sum(x is None for x in _rt) == 10,                           "⑦ exactly 10 timeouts (even idx)!"
assert [x for x in _rt if x is not None] == [i for i in range(20) if i % 2 == 1], "⑧ odd-index jeete!"

async def _fire():
    t1 = asyncio.create_task(_pr(0, 0))
    t2 = asyncio.create_task(_pr(19, 95))
    while not (t1.done() and t2.done()):
        await asyncio.sleep(0.005)
    return t1.result()["scaled"], t2.result()["scaled"]
_f1, _f2 = run(_fire())
assert (_f1, _f2) == (0.0, 85.5),                                   "⑨ create_task fire-&-collect!"

assert abs(855.0 / 20 - 42.75) < 1e-9,                              "⑩ avg exactly 42.75!"
print("🏆 10/10 ETL-IN-CHARGE BADGE — toolkit tumhaari hook me! 🔧⏬")

🏆 10/10 ETL-IN-CHARGE BADGE — toolkit tumhaari hook me! 🔧⏬


---
## ✅ SOLUTIONS 🔓

<details><summary><b>S1-S4</b></summary>

In [10]:
# S1-S4 ✅
VALS = [int(r["value"]) for r in rows]
print("vals:", VALS[:4], "...", VALS[-3:])                # [0,5,10,15] ... [85,90,95]

async def process_row(idx, val):
    await asyncio.sleep(0.01)
    return {"idx": idx, "scaled": val * 0.9, "topic": "HIGH" if val >= 50 else "LOW"}
one = run(process_row(19, 95))
print("sample:", one)                                     # idx 19, 85.5 HIGH

PROC = run(asyncio.gather(*(process_row(i, v) for i, v in enumerate(VALS))))
print("proced:", len(PROC), "rows | first:", PROC[0], "| last:", PROC[-1])

PEAK = 0
async def throttle_all():
    global PEAK
    sem = asyncio.Semaphore(2)
    live = 0
    async def one(i, v):
        nonlocal live
        global PEAK
        async with sem:
            live += 1; PEAK = max(PEAK, live)
            await asyncio.sleep(0.01)
            live -= 1
            return i
    await asyncio.gather(*(one(i, v) for i, v in enumerate(VALS)))
run(throttle_all())
print("peak concurrency:", PEAK)                          # 2 — vada nibhaya!

vals: [0, 5, 10, 15] ... [85, 90, 95]
sample: {'idx': 19, 'scaled': 85.5, 'topic': 'HIGH'}
proced: 20 rows | first: {'idx': 0, 'scaled': 0.0, 'topic': 'LOW'} | last: {'idx': 19, 'scaled': 85.5, 'topic': 'HIGH'}
peak concurrency: 2


<details><summary><b>S5-S7</b></summary>

In [11]:
# S5-S7 ✅
TIMEOUTS = []
async def safe_read(i, sleep_t):
    async def inner():
        await asyncio.sleep(sleep_t)
        return i
    try:
        return await asyncio.wait_for(inner(), timeout=0.1)
    except TimeoutError:
        TIMEOUTS.append(i)
        return None
res5 = run(asyncio.gather(*(safe_read(i, 0.4 if i % 2 == 0 else 0.02) for i in range(20))))
print("timeouts:", TIMEOUTS == [i for i in range(20) if i % 2 == 0], "| ok:",
      sum(x is not None for x in res5), "/", len(res5))

TRIO = []
async def collect_first3():
    tasks = [asyncio.create_task(process_row(i, VALS[i])) for i in range(3)]
    print("  just-created done flags:", [t.done() for t in tasks], "(sab False!)")
    for t in tasks:
        TRIO.append(await t)
run(collect_first3())
print("trio:", [(x["idx"], x["scaled"]) for x in TRIO])   # [(0,0.0),(1,4.5),(2,9.0)]

TOT = sum(p["scaled"] for p in PROC)
AVG = TOT / len(PROC)
print(f"bill: TOT={TOT} | AVG={AVG}  (exact 855.0 & 42.75!)")
# Takeaway: csv→async-map→sem-lanes→timeout-card→fire-collect→bill = full ETL chain! 🔗

timeouts: True | ok: 10 / 20
  just-created done flags: [False, False, False] (sab False!)
trio: [(0, 0.0), (1, 4.5), (2, 9.0)]
bill: TOT=855.0 | AVG=42.75  (exact 855.0 & 42.75!)


---
## 🏁 Checklist — ETL-In-Charge 🔧
| ✔ | Skill |
|---|---|
| ⬜ | csv read + clean (int-cast!) 🧹 |
| ⬜ | async processor + gather-map 🗺️ |
| ⬜ | Semaphore lanes + peak-counter 🚦 |
| ⬜ | wait_for red-card (even/odd test!) 🛡️ |
| ⬜ | create_task fire-&-collect 📝 |

**ALL GREEN?** 🎉 Last topic: **04_When_to_Use_Each** — a decision-tree for WHO-IS-THE-KING among threads vs processes
vs async; master-scenario charts for all three + golden rules! 🗺️👑